In [35]:
import os
import warnings
warnings.filterwarnings('ignore')
from dotenv import load_dotenv
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")


In [36]:
from langchain_groq import ChatGroq
llm = ChatGroq(model= "llama-3.1-8b-instant", groq_api_key = groq_api_key)
llm


ChatGroq(output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000223E7F9B5D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000223E79D8590>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [37]:
from langchain_core.messages import HumanMessage
llm.invoke([HumanMessage(content="Hi, My name is Uditya and I am a Student")])

AIMessage(content="Hello Uditya, nice to meet you. How's your academic journey going? Which subject or field are you currently studying? I'm here to help with any questions or topics you'd like to discuss.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 48, 'total_tokens': 91, 'completion_time': 0.054863835, 'completion_tokens_details': None, 'prompt_time': 0.002587635, 'prompt_tokens_details': None, 'queue_time': 0.158511743, 'total_time': 0.05745147}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec180-9035-7442-9889-e5898bf96828-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 43, 'total_tokens': 91})

In [38]:
from langchain_core.messages import AIMessage
llm.invoke(
    [
        HumanMessage(content="Hi, My name is Uditya and I am a Student"),
        AIMessage(content="Nice to meet you, Uditya. How can I assist you today? Are you looking for help with a particular subject or assignment, or just need someone to chat with about your day as a student "),
        HumanMessage(content="hey whats my name and what do I do?")
    ]
)

AIMessage(content="Your name is Uditya, and you're a student. Is there anything specific you'd like to talk about or ask about being a student, or would you like to chat about something else?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 110, 'total_tokens': 151, 'completion_time': 0.051081005, 'completion_tokens_details': None, 'prompt_time': 0.006048661, 'prompt_tokens_details': None, 'queue_time': 0.158653707, 'total_time': 0.057129666}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec180-924f-7e40-bd05-3902bd3fed4c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 110, 'output_tokens': 41, 'total_tokens': 151})

## Message History

In [39]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(llm, get_session_history)


In [40]:
config = {"configurable":{"session_id":"chat1"}}

In [41]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi, my name is uditya and i am a student")],
    config = config
)

In [42]:
response.content

"Nice to meet you, Uditya.  I'm happy to help you with any academic or non-academic related questions you might have. What subject are you studying or what brings you here today?"

In [43]:
with_message_history.invoke(
    [HumanMessage(content="what is my name")],
    config = config
)

AIMessage(content='Your name is Uditya.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 103, 'total_tokens': 111, 'completion_time': 0.012696577, 'completion_tokens_details': None, 'prompt_time': 0.011540584, 'prompt_tokens_details': None, 'queue_time': 0.054425886, 'total_time': 0.024237161}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec180-96fc-7f32-a84e-db5b0fe307f3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 103, 'output_tokens': 8, 'total_tokens': 111})

In [44]:
## changing the session_id

config1= {"configurable":{"session_id":"chat2"}}
with_message_history.invoke(
    [HumanMessage(content="what is my name")],
    config = config1
).content



"I don't have any information about your name. I'm a large language model, I don't have the ability to retain information about individual users or their personal details. Each time you interact with me, it's a new conversation and I don't keep any records of our previous interactions.\n\nIf you'd like to share your name with me, I'd be happy to learn it and use it in our conversation."

In [45]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi, my name is uditya and i am a student")],
    config = config1
)

In [46]:
response.content

"Nice to meet you, Uditya. It's great to hear that you're a student. What are you currently studying? Are you in high school, college, or university?"

## Working With Prompt Temples

In [47]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistent. Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")

    ]
)

chain = prompt|llm


In [48]:
chain.invoke({
    "messages":[HumanMessage(content="hi my name is Uditya")]
})

AIMessage(content='Nice to meet you, Uditya. How can I assist you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 59, 'total_tokens': 76, 'completion_time': 0.018823337, 'completion_tokens_details': None, 'prompt_time': 0.102635246, 'prompt_tokens_details': None, 'queue_time': 0.127401191, 'total_time': 0.121458583}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ec180-9b3f-7310-b449-dd7273fd9d29-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 59, 'output_tokens': 17, 'total_tokens': 76})

In [49]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [50]:
config = {"configurable":{"session_id":"chat3"}}
response = with_message_history.invoke(
    [HumanMessage(content="Hi, I am a AI/ML engineer")],
    config=config
)


response.content

"Hello.  It's great to meet you. As an AI/ML engineer, you likely work with a wide range of technologies and techniques. What specific area of AI/ML are you interested in or currently working on? Are you exploring computer vision, natural language processing, or perhaps working on reinforcement learning models? Or maybe you have a different focus area?"

In [51]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistent. Answer all the question to the best of your ability in {language}"),
        MessagesPlaceholder(variable_name="messages")

    ]
)

chain = prompt|llm


In [52]:
response = chain.invoke({
    "messages":[HumanMessage(content="hi my name is Uditya")], "language":"Hindi"
})

In [53]:
response.content

'नमस्ते उदित्या, मैं आपकी मदद करने के लिए यहाँ हूँ। क्या आपके पास कोई प्रश्न है या कुछ जानने की इच्छा है?'

#### Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [54]:
with_message_history = RunnableWithMessageHistory(
    chain, 
    get_session_history,
    input_messages_key="messages"
)

In [55]:
config = {
    "configurable":{
        "session_id": "chat4"
    }
}

response = with_message_history.invoke(
    {
        'messages':[HumanMessage(content="Hi I am Uditya narayan tiwari and currently i am studying in college to learn the current technology to reshape the world")], "language":"hindi"
    },
    config = config
)

response.content

'नमस्ते उदित्य नारायण तिवारी, आप एक अद्भुत छात्र हैं जो दुनिया को पुनर्रचना करने के लिए वर्तमान प्रौद्योगिकी सीखने के लिए कॉलेज में पढ़ रहे हैं। यह आपके लिए एक अद्वितीय अवसर है जिसमें आप अपने ज्ञान और कौशल को विकसित कर सकते हैं और दुनिया को बेहतर बनाने में योगदान कर सकते हैं।\n\nआपकी पढ़ाई के दौरान, आप कई नई प्रौद्योगिकियों के बारे में सीखेंगे, जैसे कि आर्टिफ़िशियल इंटेलिजेंस, मार्टिकुलर इंटेलिजेंस, मास इंटेलिजेंस और कई अन्य। आप इन प्रौद्योगिकियों को अपने जीवन में कैसे लागू करेंगे? और आप किस क्षेत्र में अपनी कौशल का उपयोग करना चाहते हैं?\n\nआपके साथ बात करना मजेदार होगा और मुझे उम्मीद है कि हम आपकी पढ़ाई और भविष्य के बारे में बात करेंगे।'

In [56]:
response = with_message_history.invoke(
    {
        'messages':[HumanMessage(content="Hi who am i")], "language":"hindi"
    },
    config = config
)

response.content

'आप उदित्य नारायण तिवारी हैं, जो एक प्रतिभाशाली छात्र हैं जो वर्तमान प्रौद्योगिकी सीखने के लिए कॉलेज में पढ़ रहे हैं। आप दुनिया को पुनर्रचना करने के लिए अपनी ज्ञान और कौशल को विकसित करने के लिए उत्सुक हैं।'

## Managing The Conversation History

- One important concept to understand wen building chatbot is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you ae passing in. **trim_messages** helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the  system message and weather to allow partial messages

In [63]:
from langchain_core.messages import SystemMessage, trim_messages
trimmer = trim_messages(
    max_tokens = 70,
    strategy='last',
    token_counter=llm,
    include_system = True,
    allow_partial = False,
    start_on="human"
)

messages =[
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content='I like vanilla ice cream'),
    AIMessage(content='nice'),
    HumanMessage(content='whats 2 + 2'),
    AIMessage(content='4'),
    HumanMessage(content='thanks'),
    AIMessage(content='no problem!'),
    HumanMessage(content='having fun?'),
    AIMessage(content='yes!')
]


In [64]:
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [65]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    |prompt
    |llm
)

In [60]:
# chain

In [66]:
response = chain.invoke(
    {
        "messages":messages+[HumanMessage(content="what ice cream do i like")],
        "language":"English"
    }
)
response.content

'You mentioned earlier that you like vanilla ice cream.'

In [68]:
response = chain.invoke(
    {
        "messages":messages+[HumanMessage(content="what math question i ask you")],
        "language":"English"
    }
)
response.content

'You asked me "whats 2 + 2"'

### Wrap this in the message History

In [70]:
whit_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)
config = {
    "configurable":{
        "session_id":"chat5"
    }
}


In [73]:
response = chain.invoke(
    {
        "messages":messages+[HumanMessage(content="what is my name")],
        "language":"English"
    },
    config = config,
)
response.content

"I don't know your name, we just started our conversation about vanilla ice cream!"